# HydrAI: ML Training Data Generation

This notebook generates training datasets for ML surrogate models by running multiple PFR simulations with varied parameters.

## Overview

The data generation process:
1. **Parameter Sweep**: Varies 6 key parameters (temperature, pressure, geometry, flow, heat flux)
2. **Multiple Reactants**: Supports ethane, propane, naphtha, n-hexane
3. **Efficient Collection**: Disables plots/CSV exports for speed
4. **Periodic Saves**: Saves data incrementally to prevent loss
5. **Rich Features**: Collects 245+ features per simulation point

## Features
- Configurable parameter ranges
- **Latin Hypercube Sampling (LHS)** for efficient parameter space coverage
- Random or full-grid sampling alternatives
- Progress tracking and time estimates
- Automatic data validation
- Metadata export for reproducibility

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime

# Get project root directory
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/
# (our package) instead of the actual cantera library
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Suppress all Cantera/SUNDIALS warnings and messages
import warnings
import logging
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='.*rank.*')
warnings.filterwarnings('ignore', message='.*CVode.*')
warnings.filterwarnings('ignore', message='.*SUNDIALS.*')
warnings.filterwarnings('ignore', message='.*WARNING.*')
logging.getLogger('cantera').setLevel(logging.CRITICAL)
logging.getLogger('sundials').setLevel(logging.CRITICAL)

# Add src to path (after cantera is imported)
sys.path.insert(0, str(project_root / 'src'))

from src.ml.data_generation import TrainingDataGenerator

print("HydrAI: ML Training Data Generation")
print("=" * 60)
print(f"Project root: {project_root}")

# Run control flags (plots and metadata)
IF_SHOW_PLOTS = False   # Show training-space plots in notebook
IF_SAVE_PLOTS = False  # Save training-space plots to output_dir
IF_SAVE_METADATA = False  # Save metadata JSON when generating dataset
IF_SAVE_TRAINING_DATA = False  # Save training data (partial + final pkl/csv) to output_dir


## Step 1: Configuration

Configure the data generation parameters. You can either:
- Load from a JSON config file
- Set parameters directly in the notebook

**Sampling method**: Use `sampling_method='latin_hypercube'` for Latin Hypercube Sampling (LHS), which gives better coverage of the parameter space with fewer runs than random or grid sampling.

In [ ]:
# Option 1: Load from JSON config file
USE_CONFIG_FILE = True  # Set to False to use manual configuration below
CONFIG_FILE = 'configs/ml_data_generation_config.json'

if USE_CONFIG_FILE and Path(CONFIG_FILE).exists():
    with open(CONFIG_FILE, 'r') as f:
        config = json.load(f)
    print(f"[OK] Loaded configuration from: {CONFIG_FILE}")
    
    # Extract parameters
    REACTANTS = config.get('reactants', ['ethane'])
    MAX_COMBINATIONS = config.get('max_combinations_per_reactant', 100)
    OUTPUT_DIR = config.get('output_dir', 'data/training')
    SAVE_INTERVAL = config.get('save_interval', 10)
    RANDOM_SAMPLE = config.get('random_sample', True)
    PARAM_RANGES = config.get('parameter_ranges', {})
    SAMPLING_METHOD = config.get('sampling_method', 'latin_hypercube')
    LHS_SEED = config.get('lhs_seed', 42)
    RANDOM_SAMPLE_BOUNDS = config.get('random_sample_bounds', None)
    
    print(f"  - Reactants: {REACTANTS}")
    print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
    print(f"  - Output directory: {OUTPUT_DIR}")
    print(f"  - Sampling method: {SAMPLING_METHOD}")
    print(f"  - Random sampling (if method=random): {RANDOM_SAMPLE}")
else:
    # Option 2: Manual configuration
    REACTANTS = ['ethane', 'propane']  # List of reactants to use
    MAX_COMBINATIONS = 100  # Maximum parameter combinations per reactant
    OUTPUT_DIR = 'data/training'  # Where to save training data
    SAVE_INTERVAL = 10  # Save every N simulations
    RANDOM_SAMPLE = True  # Used when sampling_method='random'
    SAMPLING_METHOD = 'latin_hypercube'  # 'random' | 'full_grid' | 'latin_hypercube'
    LHS_SEED = 42
    RANDOM_SAMPLE_BOUNDS = None
    
    # Parameter ranges: [min, max, n_points]
    PARAM_RANGES = {
        'temperature_K': [800, 1200, 10],
        'pressure_bar': [1.5, 3.0, 8],
        'length_m': [3.0, 7.0, 6],
        'diameter_mm': [20.0, 40.0, 5],
        'mass_flow_rate_kgps': [0.05, 0.10, 6],
        'heat_flux_Wm2': [100000, 200000, 5]
    }
    print("[OK] Using manual configuration")
    print(f"  - Reactants: {REACTANTS}")
    print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
    print(f"  - Sampling method: {SAMPLING_METHOD}")

print(f"\nParameter Ranges:")
for param, values in PARAM_RANGES.items():
    if isinstance(values, list) and len(values) == 3:
        print(f"  - {param}: {values[0]} to {values[1]} ({values[2]} points)")
    else:
        print(f"  - {param}: {values}")

## Step 2: Initialize Data Generator

Create the data generator with your configuration.

In [ ]:
# Initialize generator
generator = TrainingDataGenerator(output_dir=OUTPUT_DIR, disable_plots=True)

# Update parameter ranges from config
if PARAM_RANGES:
    for key, value in PARAM_RANGES.items():
        if isinstance(value, list) and len(value) == 3:
            # Convert [min, max, n_points] to numpy array
            generator.param_ranges[key] = np.linspace(value[0], value[1], value[2])
            print(f"[OK] Updated {key}: {len(generator.param_ranges[key])} points")

# Calculate total combinations
total_combinations = generator._calculate_total_combinations()
print(f"\n[OK] Data generator initialized")
print(f"  - Output directory: {generator.output_dir}")
print(f"  - Total possible combinations: {total_combinations:,}")
print(f"  - Will generate: {len(REACTANTS) * MAX_COMBINATIONS:,} simulations")

### Step 2.1: Visualize training space (sampling preview)

Preview how the parameter space will be explored: 1D marginals (uniformity) and 2D pairs (space filling).

In [1]:
# Generate the same parameter combinations that will be used in training data generation
_method = (str(SAMPLING_METHOD).strip().lower() if SAMPLING_METHOD else 'random')
if _method == 'latin': _method = 'latin_hypercube'
if _method == 'latin_hypercube':
    param_combinations = generator.generate_parameter_combinations_lhs(
        MAX_COMBINATIONS, random_sample_bounds=RANDOM_SAMPLE_BOUNDS, seed=LHS_SEED)
else:
    param_combinations = generator.generate_parameter_combinations(
        max_combinations=MAX_COMBINATIONS, random_sample=(_method == 'random'),
        random_sample_bounds=RANDOM_SAMPLE_BOUNDS)

# Build DataFrame for plotting (use display units: bar, mm)
rows = []
for p in param_combinations:
    rows.append({
        'temperature_K': p['temperature_K'],
        'pressure_bar': p['pressure_Pa'] / 1e5,
        'length_m': p['length_m'],
        'diameter_mm': p['diameter_m'] * 1000,
        'mass_flow_rate_kgps': p['mass_flow_rate_kgps'],
        'heat_flux_Wm2': p['heat_flux_Wm2']})
df_design = pd.DataFrame(rows)
param_names = list(df_design.columns)

# 1D marginals: uniformity of exploration per parameter
fig1, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for i, col in enumerate(param_names):
    axes[i].hist(df_design[col], bins=min(30, len(df_design)//5), edgecolor='black', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
plt.suptitle('Training space: 1D marginals (sampling preview)', fontsize=12)
plt.tight_layout()
if IF_SAVE_PLOTS:
    generator.output_dir.mkdir(parents=True, exist_ok=True)
    fig1.savefig(generator.output_dir / 'training_space_1d_preview.png', dpi=150, bbox_inches='tight')
if IF_SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig1)

# 2D pairwise scatter: space-filling quality (selected pairs)
pairs = [('temperature_K', 'pressure_bar'), ('length_m', 'diameter_mm'), ('mass_flow_rate_kgps', 'heat_flux_Wm2'),
         ('temperature_K', 'length_m'), ('pressure_bar', 'heat_flux_Wm2'), ('diameter_mm', 'mass_flow_rate_kgps')]
fig2, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for i, (x, y) in enumerate(pairs):
    axes[i].scatter(df_design[x], df_design[y], alpha=0.5, s=15)
    axes[i].set_xlabel(x)
    axes[i].set_ylabel(y)
    axes[i].set_title(f'{x} vs {y}')
plt.suptitle('Training space: 2D coverage (sampling preview)', fontsize=12)
plt.tight_layout()
if IF_SAVE_PLOTS:
    generator.output_dir.mkdir(parents=True, exist_ok=True)
    fig2.savefig(generator.output_dir / 'training_space_2d_preview.png', dpi=150, bbox_inches='tight')
if IF_SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig2)

print(f"[OK] Design: {len(df_design)} combinations × {len(param_names)} parameters.")
print(f"     Good LHS/random: 1D histograms fairly uniform, 2D points well spread.")

NameError: name 'SAMPLING_METHOD' is not defined

## Step 3: Generate Training Data

Run the data generation process. This will:
- Run multiple PFR simulations with varied parameters
- Collect features and targets at each simulation point
- Save data incrementally to prevent loss
- Generate metadata for reproducibility

**Note**: This may take 5-30 minutes depending on the number of simulations.

In [ ]:
# Generate dataset
print("=" * 60)
print("Starting Training Data Generation...")
print("=" * 60)
print(f"Reactants: {REACTANTS}")
print(f"Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"Sampling method: {SAMPLING_METHOD}")
print(f"Save interval: {SAVE_INTERVAL} simulations")
print("=" * 60)

start_time = time.time()

dataset = generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=MAX_COMBINATIONS,
    random_sample=RANDOM_SAMPLE,
    save_interval=SAVE_INTERVAL,
    sampling_method=SAMPLING_METHOD,
    lhs_seed=LHS_SEED,
    save_metadata=IF_SAVE_METADATA,
    save_training_data=IF_SAVE_TRAINING_DATA
)

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n[OK] Data generation completed!")
print(f"  - Total time: {hours:02d}:{minutes:02d}:{seconds:02d}")

# Calculate actual number of simulations completed
# The dataset has multiple rows per simulation (one per reactor step)
# We get the actual count from the metadata file
if dataset is not None and len(dataset) > 0:
    import glob
    metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')), reverse=True)
    if metadata_files:
        with open(metadata_files[0], 'r') as f:
            metadata = json.load(f)
        n_simulations = metadata.get('total_simulations', len(REACTANTS) * MAX_COMBINATIONS)
    else:
        n_simulations = len(REACTANTS) * MAX_COMBINATIONS
    
    print(f"  - Simulations completed: {n_simulations}")
    print(f"  - Data points collected: {len(dataset):,}")
    if n_simulations > 0:
        print(f"  - Average time per simulation: {elapsed_time / n_simulations:.2f} seconds")
        print(f"  - Data points per simulation: {len(dataset) / n_simulations:.1f}")
    else:
        print("  - Average time: N/A (no simulations completed)")
else:
    print("  - Average time: N/A (no data collected)")


## Step 4: Dataset Summary

Display information about the generated dataset.

In [ ]:
if dataset is not None:
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Shape: {dataset.shape[0]:,} rows × {dataset.shape[1]} columns")
    print(f"\nColumn Categories:")
    
    # Count different types of columns
    input_cols = [c for c in dataset.columns if c in ['temperature_K', 'pressure_Pa', 
                                                       'reactor_length_m', 'reactor_diameter_m',
                                                       'mass_flow_rate_kgps', 'heat_flux_Wm2', 'z_position_m']]
    output_cols = [c for c in dataset.columns if c in ['velocity_ms', 'density_kgm3']]
    species_cols = [c for c in dataset.columns if c.startswith('Y_') or c.startswith('X_')]
    
    print(f"  - Input features: {len(input_cols)}")
    print(f"  - Output targets: {len(output_cols)}")
    print(f"  - Species data: {len(species_cols)} (mass + mole fractions)")
    print(f"  - Total columns: {len(dataset.columns)}")
    
    print(f"\nData Statistics:")
    print(dataset[['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3']].describe())
    
    print(f"\n[OK] Dataset ready for ML training!")
    print(f"  - Saved to: {generator.output_dir}")
else:
    print("[ERROR] No dataset generated!")

### Step 4.1: Training space coverage (from generated data)

Visualize how well the parameter space was actually explored using the generated dataset (one point per simulation).

In [ ]:
if dataset is not None and len(dataset) > 0:
    # One row per simulation (unique input conditions)
    input_cols = ['initial_temperature_K', 'initial_pressure_Pa', 'reactor_length_m', 'reactor_diameter_m',
                  'mass_flow_rate_kgps', 'heat_flux_Wm2']
    df_unique = dataset[input_cols].drop_duplicates()
    df_unique = df_unique.rename(columns={'initial_temperature_K': 'temperature_K', 'initial_pressure_Pa': 'pressure_Pa'})
    df_unique['pressure_bar'] = df_unique['pressure_Pa'] / 1e5
    df_unique['diameter_mm'] = df_unique['reactor_diameter_m'] * 1000
    plot_cols = ['temperature_K', 'pressure_bar', 'reactor_length_m', 'diameter_mm', 'mass_flow_rate_kgps', 'heat_flux_Wm2']
    df_plot = df_unique[plot_cols]

    # 1D marginals from actual data
    fig1, axes = plt.subplots(2, 3, figsize=(12, 7))
    axes = axes.flatten()
    for i, col in enumerate(plot_cols):
        axes[i].hist(df_plot[col], bins=min(30, max(5, len(df_plot)//5)), edgecolor='black', alpha=0.7, color='steelblue')
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
    plt.suptitle('Training space: 1D marginals (from generated data)', fontsize=12)
    plt.tight_layout()
    if IF_SAVE_PLOTS:
        generator.output_dir.mkdir(parents=True, exist_ok=True)
        fig1.savefig(generator.output_dir / 'training_space_1d_from_data.png', dpi=150, bbox_inches='tight')
    if IF_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig1)

    # 2D pairwise scatter from actual data
    pairs = [('temperature_K', 'pressure_bar'), ('reactor_length_m', 'diameter_mm'), ('mass_flow_rate_kgps', 'heat_flux_Wm2'),
             ('temperature_K', 'reactor_length_m'), ('pressure_bar', 'heat_flux_Wm2'), ('diameter_mm', 'mass_flow_rate_kgps')]
    fig2, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, (x, y) in enumerate(pairs):
        axes[i].scatter(df_plot[x], df_plot[y], alpha=0.5, s=15, c='steelblue')
        axes[i].set_xlabel(x)
        axes[i].set_ylabel(y)
        axes[i].set_title(f'{x} vs {y}')
    plt.suptitle('Training space: 2D coverage (from generated data)', fontsize=12)
    plt.tight_layout()
    if IF_SAVE_PLOTS:
        generator.output_dir.mkdir(parents=True, exist_ok=True)
        fig2.savefig(generator.output_dir / 'training_space_2d_from_data.png', dpi=150, bbox_inches='tight')
    if IF_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig2)

    print(f"[OK] Unique simulations in dataset: {len(df_plot):,}. Check that 1D/2D coverage matches your sampling method.")
else:
    print("Run data generation first to see training space coverage from generated data.")

## Summary

Your training data has been generated and visualized.

1. **Sampling preview (Step 2.1)**: 1D marginals and 2D pairs show the *planned* training space; for LHS expect fairly uniform 1D and good 2D spread.
2. **Coverage from data (Step 4.1)**: Same plots from *actual* generated data; confirm they match the preview (failed runs may reduce coverage).
3. **Parameter coverage**: Ensure good coverage of the parameter space for ML.
4. **Correlations**: Identify strongly correlated features (may need feature engineering)
5. **Data Quality**: Verify no missing/infinite values or excessive outliers
6. **Species Diversity**: Ensure all important species are present in sufficient concentrations

### Next Steps:
1. **Train ML Models**: Use the generated dataset to train surrogate models
2. **Feature Engineering**: Consider transformations based on correlation analysis
3. **Data Augmentation**: Generate more data if coverage is insufficient
4. **Validation Split**: Reserve 20-30% of data for model validation